<a href="https://colab.research.google.com/github/DDarion/compressed-gaussians-colab/blob/main/ECNG_3020_3DGS%20-%20Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Set-up
## Confirming the GPU being used

In [6]:
!nvidia-smi

Fri Nov 20 14:33:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Version comparions

In [7]:
# nvcc and pytorch cuda version
%cd /content/gaussian-splatting
!nvcc --version
import torch
print(torch.__version__, torch.version.cuda)
!pip install -v --no-build-isolation submodules/diff-gaussian-rasterization 2>&1 | tail -60

/content/gaussian-splatting
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
2.14.1+cu130 13.0
  running dist_info
  creating /tmp/pip-modern-metadata-5y2rp7o_/diff_gaussian_rasterization.egg-info
  writing /tmp/pip-modern-metadata-5y2rp7o_/diff_gaussian_rasterization.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-5y2rp7o_/diff_gaussian_rasterization.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-modern-metadata-5y2rp7o_/diff_gaussian_rasterization.egg-info/top_level.txt
  writing manifest file '/tmp/pip-modern-metadata-5y2rp7o_/diff_gaussian_rasterization.egg-info/SOURCES.txt'
  W1120 14:33:07.566000 27069 torch/utils/cpp_extension.py:786] Attempted to use ninja as the BuildExtension backend but we could not find ninja.. Falling back to using the slow distutils backend.
  reading

# Setting up baseline Kerbl model
## Point cloud

In [8]:
# mounting Drive first so outputs survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
RUN = "/content/drive/MyDrive/3dgs_outputs/train_baseline"   # where the output goes

# getting the code from kerbl repo
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
!pip install -q plyfile

# patching and building the CUDA extensions
%cd /content/gaussian-splatting
!grep -q "<cstdint>" submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h || sed -i '0,/#pragma once/s//#pragma once\n#include <cstdint>/' submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
!head -15 submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h
!pip install --no-build-isolation submodules/diff-gaussian-rasterization
!pip install --no-build-isolation submodules/simple-knn
!python -c "import diff_gaussian_rasterization, simple_knn; print('ok')"

# getting the dataset
%cd /content/gaussian-splatting
# below is the main dataset
!wget -nc -q https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip
# below is the mirror dataset
# !wget https://huggingface.co/camenduru/gaussian-splatting/resolve/main/tandt_db.zip
!unzip -oq tandt_db.zip
!date # to show the date when the model was trained
!ls tandt

# training the model (--eval holds out test views, -m saves into Drive)
!python train.py -s /content/gaussian-splatting/tandt/train --eval -m {RUN}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
fatal: destination path 'gaussian-splatting' already exists and is not an empty directory.
/content/gaussian-splatting
/*
 * Copyright (C) 2023, Inria
 * GRAPHDECO research group, https://team.inria.fr/graphdeco
 * All rights reserved.
 *
 * This software is free for non-commercial, research and evaluation use 
 * under the terms of the LICENSE.md file.
 *
 * For inquiries contact  george.drettakis@inria.fr
 */

#pragma once
#include <cstdint>

#include <iostream>
Processing ./submodules/diff-gaussian-rasterization
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp313-cp313-linux_x86_64.whl size=3643909 sha256=10665aa863625ae8877e845bbe964d99258c5943683a68617bee59ba0c0575a1
  Stored in directory: /tmp/pip-ephem-wheel-cache-puiw8jiq/wheels/49/ab/50/4f594db59dd03

## Output Metrics

In [9]:
# you NEED to run the cell before since {RUN} needs to be defined within the
# same session
%cd /content/gaussian-splatting
!pip install -q lpips
!python render.py -m {RUN}
!python metrics.py -m {RUN}
!ls {RUN}

/content/gaussian-splatting
Looking for config file in /content/drive/MyDrive/3dgs_outputs/train_baseline/cfg_args
Config file found: /content/drive/MyDrive/3dgs_outputs/train_baseline/cfg_args
Rendering /content/drive/MyDrive/3dgs_outputs/train_baseline
Loading trained model at iteration 30000 [20/11 15:27:45]
------------LLFF HOLD------------- [20/11 15:27:47]
Reading camera 301/301 [20/11 15:27:47]
Loading Training Cameras [20/11 15:27:47]
Loading Test Cameras [20/11 15:27:52]
Rendering progress: 100% 263/263 [02:12<00:00,  1.99it/s]
Rendering progress: 100% 38/38 [00:18<00:00,  2.04it/s]

Scene: /content/drive/MyDrive/3dgs_outputs/train_baseline
Method: ours_30000
Metric evaluation progress: 100% 38/38 [01:04<00:00,  1.71s/it]
  SSIM :    0.8208814
  PSNR :   22.0750027
  LPIPS:    0.1961879

cameras.json					     input.ply
cfg_args					     per_view.json
events.out.tfevents.1794788962.f41f0991ea15.14226.0  point_cloud
events.out.tfevents.1795179775.a6c2142729c8.3272.0   results.js

## Model size and Gaussian count

In [10]:
# you NEED to run the cell before since {RUN} needs to be defined within the
# same session
!ls -lh {RUN}/point_cloud/iteration_30000/point_cloud.ply
!head -c 2000 {RUN}/point_cloud/iteration_30000/point_cloud.ply | strings | grep "element vertex"

-rw------- 1 root root 258M Nov 20 15:27 /content/drive/MyDrive/3dgs_outputs/train_baseline/point_cloud/iteration_30000/point_cloud.ply
element vertex 1089740
